# Notebook 05 — Locked test evaluation, advanced analyses and 20 figures

**Purpose.** One-time evaluator-role scoring of the frozen inference bundles on the locked test partition: raw and temperature-calibrated probabilities, patient/row KL, hard metrics, calibration, disagreement and referral, paired component bootstrap, strata, the six-condition stress suite, evidence-deletion audits, latency, resources, and figures V01–V20. **Inputs:** frozen final runs, protocol lock. **Outputs:** `private/evaluation/<protocol_hash>/`, `results/aggregate/`, `figures/`.

In [1]:
import os, sys, json, subprocess
from pathlib import Path
REPO = Path.cwd().resolve() if (Path.cwd() / "src" / "cape_eeg").exists() else Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("CAPE_ROOT", str(REPO.parent)); os.environ.setdefault("HMS_DATA_ROOT", os.environ["CAPE_ROOT"])
os.environ["PYTHONWARNINGS"] = "ignore"
from cape_eeg.paths import resolve_workspace, redact
from cape_eeg.status import read_json, Ledger
ws = resolve_workspace()
def run(cmd, **kw):
    """Run a repository script as a bounded subprocess; prints filtered output (no secrets, no identifiers)."""
    p = subprocess.run([sys.executable, str(REPO / "scripts" / cmd[0]), *cmd[1:]], capture_output=True, text=True, env=os.environ, **kw)
    for line in (p.stdout + p.stderr).splitlines():
        if line.strip() and not any(w in line for w in ("Warning", "warn", "Found GPU", "Minimum and", "(8.0)")):
            print(line)
    if p.returncode != 0:
        raise RuntimeError(f"{cmd[0]} exited with {p.returncode}")
print("repo:", redact(REPO, ws)); print("workspace root:", redact(ws.root, ws)); print("data root:", redact(ws.data, ws)); print("private:", redact(ws.private, ws))


repo: $CAPE_ROOT/cape-eeg
workspace root: $CAPE_ROOT
data root: $CAPE_ROOT
private: $CAPE_ROOT/private


## Final-test gate and one-time evaluation

In [2]:
lock = read_json(ws.manifests / 'protocol_lock.json'); out = ws.evaluation / lock['protocol_hash']
if not (out / 'summary.json').exists():
    run(['final_evaluate.py'])
else:
    print('locked evaluation already exists for protocol', lock['protocol_hash'], '- not re-run')
print(open(ws.manifests / 'test_access_log.jsonl').read()[:600])

locked evaluation already exists for protocol 380269a4a6db7334 - not re-run
{"run": "final_P_s101_4455edaa_2406ce59_cf867030", "partitions": ["calibration_t", "calibration_p", "test"], "timestamp": "2026-09-06T19:45:41+00:00", "protocol_hash": "380269a4a6db7334"}
{"run": "final_P_s202_4455edaa_2406ce59_cf867030", "partitions": ["calibration_t", "calibration_p", "test"], "timestamp": "2026-09-06T19:45:48+00:00", "protocol_hash": "380269a4a6db7334"}
{"run": "final_P_s303_4455edaa_2406ce59_cf867030", "partitions": ["calibration_t", "calibration_p", "test"], "timestamp": "2026-09-06T19:45:54+00:00", "protocol_hash": "380269a4a6db7334"}
{"run": "final_B3_s101_4455edaa_2406


## Primary result

In [3]:
s = read_json(out / 'summary.json'); pr = s['primary']['primary_calibrated']
print('Delta patient/component-mean KL (P - %s), three seeds, calibrated: %+.4f [%+.4f, %+.4f] -> %s' % (lock['comparator'], pr['point_estimate'], pr['ci_low'], pr['ci_high'], pr['decision']))
print('per-seed P:', [round(x, 4) for x in pr['per_seed_a']], '| comparator:', [round(x, 4) for x in pr['per_seed_b']], '| relative change %.1f%%' % (100 * pr['relative_change']))
import pandas as pd; print(pd.read_csv(ws.results_public / 'table2_test_metrics_summary.csv').to_string(index=False))

Delta patient/component-mean KL (P - B3), three seeds, calibrated: +0.0657 [+0.0334, +0.0973] -> inferior
per-seed P: [0.9235, 0.9546, 0.9244] | comparator: [0.8776, 0.8564, 0.8715] | relative change 7.6%
method calibration  patient_kl_mean  patient_kl_sd  row_kl_mean  row_kl_sd  macro_auroc_mean  balanced_accuracy_mean  soft_top_label_ece_mean  expected_brier_mean  kl_votes_10plus_mean  params  primary_delta_vs_comparator  primary_ci_low  primary_ci_high primary_decision
    B3  calibrated         0.868487       0.010946     0.806574   0.004177          0.884214                0.576084                 0.035971             0.577637              0.665607 1524150                          NaN             NaN              NaN              NaN
    B3         raw         0.942466       0.023088     0.868690   0.012360          0.883774                0.576084                 0.107178             0.592235              0.754429 1524150                          NaN             NaN              

## Strata, robustness, evidence and referral

In [4]:
print(pd.read_csv(ws.results_public / 'table4_strata_robustness_referral.csv').to_string(index=False, max_colwidth=60))

              analysis                   item     delta    ci_low  ci_high  n_rows  n_groups  kl_candidate  kl_comparator status low_support                                                        extra
               stratum                votes_1  0.122762 -0.032063 0.289077   510.0      28.0      0.939412       0.816650   PASS        True                                                          NaN
               stratum              votes_2-4  0.056778  0.012485 0.098636 11990.0     353.0      1.057154       1.000376   PASS       False                                                          NaN
               stratum              votes_5-9  0.017513 -0.042737 0.073654   315.0      56.0      0.490964       0.473452   PASS       False                                                          NaN
               stratum              votes_10+  0.078821  0.027128 0.128859  8490.0     239.0      0.744429       0.665607   PASS       False                                                    

## Twenty figures

In [5]:
run(['make_figures.py'])
man = read_json(ws.figures_public / 'manifest.json'); print({f['figure_id']: f['status'] for f in man})

V01  PASS     Cohort and exclusion flow                                2 files    0.3s  
V02  PASS     Partition independence matrix                            2 files    0.5s  
V03  PASS     Six-class label distribution                             4 files    0.7s  
V04  PASS     Vote count and ambiguity distribution                    2 files    0.5s  
V05  PASS     Temporal alignment and foveated allocation               2 files    0.7s  
V06  PASS     Missingness and signal-quality audit                     4 files    0.8s  
V07  PASS     Byte-budget representation comparison                    2 files    0.3s  
V08  PASS     Learning curves and completed training exposure          2 files    0.3s  
V09  PASS     Primary paired patient-KL improvement                    4 files    0.6s  
V10  PASS     One-vs-rest ROC curves                                   4 files   59.8s  
V11  PASS     One-vs-rest precision-recall curves                      4 files   51.5s  
V12  PASS     Confusi

## Claims checklist

In [6]:
print('Primary decision:', pr['decision']); print('Confirmatory comparison count: 1 (P vs locked comparator). All other comparisons are exploratory or development-only.')
print('Resources:', {k: s['resources'][k] for k in ['gpu_hours_total','n_gpu_jobs','failed_or_incomplete_jobs']})

Primary decision: inferior
Confirmatory comparison count: 1 (P vs locked comparator). All other comparisons are exploratory or development-only.
Resources: {'gpu_hours_total': 0.722194269100825, 'n_gpu_jobs': 17, 'failed_or_incomplete_jobs': 0}
